# Forex Trading Strategy Analysis

This notebook analyzes EUR/USD Forex trading data to identify profitable trading strategies based on various technical indicators and market conditions.

In [117]:
# Import required libraries
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# ================== DATA LOADING & PREPROCESSING ==================

def load_and_clean_data(filepath='./eurusd.csv'):
    """
    Load EUR/USD data from CSV and clean NaN values.
    
    Args:
        filepath (str): Path to the CSV file containing trading data
    
    Returns:
        pd.DataFrame: Cleaned dataframe with trading data
    """
    df = pd.read_csv(filepath)
    
    # Define columns that should have NaN replaced with 0
    columns_to_fillna = [
        'SL', 'TP', 'SL 5M CC', 'SL 5M Stop', 
        'SL Breakout', 'Hours Until News', 'Extra'
    ]
    
    # Fill NaN values to prevent calculation errors
    for col in columns_to_fillna:
        if col in df.columns:
            df[col] = df[col].fillna(0)
    
    return df

def percentage(value, total):
    """
    Calculate percentage of value over total.
    
    Args:
        value (int/float): The numerator value
        total (int/float): The denominator value
    
    Returns:
        str: Formatted percentage string (e.g., "45.5%")
    """
    if total == 0:
        return "0.0%"
    return f"{value / total * 100:.1f}%"

# Load the data
df = load_and_clean_data()

# ================== PROFITABLE TRADES ANALYSIS ==================

def calculate_profitable_trades(df):
    """
    Calculate profitability statistics for different trading strategies.
    
    Args:
        df (pd.DataFrame): Trading data
    
    Returns:
        pd.DataFrame: Profitability statistics for various strategies
    """
    # Define trading strategies with their filters
    strategies = {
        'Total': lambda d: d[(d['SL'] != d['Pullback'])],
        'With Extra': lambda d: d[((d['SL'] != d['Pullback']) | 
                                  ((d['SL'] == d['Pullback']) & (d['Extra'].notna())))],
        'With EMA': lambda d: d[(d['SL'] != d['Pullback']) & (d['EMA'] == d['Direction'])],
        'Against EMA': lambda d: d[(d['SL'] != d['Pullback']) & (d['EMA'] != d['Direction'])],
        'BOS Trades': lambda d: d[(d['SL'] != d['Pullback']) & (d['BOS/CH'] == 'BOS')],
        'CH Trades': lambda d: d[(d['SL'] != d['Pullback']) & (d['BOS/CH'] == 'CH')],
        'With EMA & BOS': lambda d: d[(d['SL'] != d['Pullback']) & 
                                      (d['EMA'] == d['Direction']) & (d['BOS/CH'] == 'BOS')],
        'With EMA & CH': lambda d: d[(d['SL'] != d['Pullback']) & 
                                     (d['EMA'] == d['Direction']) & (d['BOS/CH'] == 'CH')],
    }
    
    results = {'Data': list(strategies.keys())}
    
    # Calculate for each Risk-Reward Ratio
    for rrr in [1, 2, 3]:
        rrr_results = []
        for strategy_name, filter_func in strategies.items():
            filtered = filter_func(df)
            
            if strategy_name == 'With Extra':
                # Special handling for 'With Extra' strategy
                profitable = filtered[filtered['TP'] >= (filtered['SL'] + filtered['Extra'].fillna(0)) * rrr]
            else:
                profitable = filtered[filtered['TP'] >= filtered['SL'] * rrr]
            
            rrr_results.append(percentage(len(profitable), len(df)))
        
        results[f'1:{rrr} RRR'] = rrr_results
    
    # Create DataFrame and sort by best 1:1 RRR performance
    df_results = pd.DataFrame(results)
    df_results['Value_num'] = df_results['1:1 RRR'].str.rstrip('%').astype(float)
    return df_results.sort_values('Value_num', ascending=False).drop(columns='Value_num').reset_index(drop=True)

# Display profitable trades analysis
display(HTML("<h2>Profitable Trades</h2><p>What are simple trading ideas that are profitable?</p>"))
df_profitable = calculate_profitable_trades(df)
display(df_profitable)

,Data,1:1 RRR,1:2 RRR,1:3 RRR
0,With Extra,51.0%,32.7%,26.5%
1,Total,38.8%,24.5%,20.4%
2,BOS Trades,25.5%,17.3%,15.3%
3,With EMA,23.5%,14.3%,11.2%
4,With EMA & BOS,18.4%,12.2%,10.2%
5,Against EMA,15.3%,10.2%,9.2%
6,CH Trades,13.3%,7.1%,5.1%
7,With EMA & CH,5.1%,2.0%,1.0%


In [118]:
# ================== ENTRY TIMING ANALYSIS ==================

def analyze_entry_timing(df):
    """
    Analyze different entry timing strategies and their success rates.
    
    This function compares various entry methods:
    - 1M Confirmation Candle: Entry on 1-minute candle confirmation
    - 5M Confirmation Candle: Entry on 5-minute candle confirmation  
    - 5M Stop: Entry with 5-minute stop loss level
    - 5M Breakout: Entry on 5-minute breakout signal
    
    Args:
        df (pd.DataFrame): Trading data with entry signals
    
    Returns:
        pd.DataFrame: Entry timing statistics sorted by success rate
    """
    # Calculate winning trades for each entry method
    entry_methods = {
        '1M Confirmation Candle': {
            'filter': lambda d: d['SL'] != 0,
            'profitable': lambda d: d[(d['SL'] != d['Pullback']) & (d['TP'] > d['SL'])],
            'sl_col': 'SL'
        },
        '5M Confirmation Candle': {
            'filter': lambda d: d['SL 5M CC'] != 0,
            'profitable': lambda d: d[(d['SL'] != d['Pullback']) & (d['TP'] > d['SL 5M CC'])],
            'sl_col': 'SL 5M CC'
        },
        '5M Stop': {
            'filter': lambda d: d['SL 5M Stop'] != 0,
            'profitable': lambda d: d[(d['SL'] != d['Pullback']) & (d['TP'] > d['SL 5M Stop'])],
            'sl_col': 'SL 5M Stop'
        },
        '5M Breakout': {
            'filter': lambda d: d['SL Breakout'] != 0,
            'profitable': lambda d: d[(d['SL'] != d['Pullback']) & (d['TP'] > d['SL Breakout'])],
            'sl_col': 'SL Breakout'
        }
    }
    
    results = []
    for method_name, method_config in entry_methods.items():
        # Get relevant trades for this method
        relevant_trades = df[method_config['filter'](df)]
        profitable_trades = method_config['profitable'](df)
        total_trades = len(relevant_trades)
        wins = len(profitable_trades)
        losses = total_trades - wins
        
        # Calculate metrics for different scenarios
        sl_col = method_config['sl_col']
        
        # With Extra calculation
        with_extra_filter = ((df['SL'] != df['Pullback']) | (df['Extra'] != 0))
        with_extra_profitable = df[with_extra_filter & (df['TP'] > df[sl_col])]
        
        # 1:3 RRR with Extra
        rrr3_with_extra = df[with_extra_filter & (df['TP'] > df[sl_col] * 3)]
        
        results.append({
            'Idea': method_name,
            'Count': f"{wins}W - {losses}L",
            'Percentage': percentage(wins, total_trades),
            'With Extra': percentage(len(with_extra_profitable), total_trades),
            'Extra & 1:3 RRR': percentage(len(rrr3_with_extra), total_trades)
        })
    
    # Convert to DataFrame and sort by win percentage
    df_results = pd.DataFrame(results)
    df_results['Value_num'] = df_results['Percentage'].str.rstrip('%').astype(float)
    return df_results.sort_values('Value_num', ascending=False).drop(columns='Value_num').reset_index(drop=True)

# Display entry timing analysis
display(HTML("<h2>When To Enter</h2><p>Following market structure, price taps order block. This is when the signal is received. Now - when to enter the trade?</p>"))
df_entry_timing = analyze_entry_timing(df)
display(df_entry_timing)

,Idea,Count,Percentage,With Extra,Extra & 1:3 RRR
0,5M Stop,31W - 32L,49.2%,66.7%,38.1%
1,5M Breakout,30W - 35L,46.2%,63.1%,40.0%
2,5M Confirmation Candle,31W - 40L,43.7%,59.2%,38.0%
3,1M Confirmation Candle,38W - 60L,38.8%,50.0%,28.6%


In [119]:
# ================== BOS WITH 1M CONFIRMATION CANDLE ANALYSIS ==================

def analyze_bos_1m_cc(df):
    """
    Analyze Break of Structure (BOS) trades using 1M Confirmation Candle (standard SL column).
    
    This is the original BOS analysis using the standard SL column which represents
    1-minute confirmation candle entries.
    
    Args:
        df (pd.DataFrame): Trading data with BOS/CH indicators
    
    Returns:
        pd.DataFrame: BOS combination statistics sorted by performance
    """
    # Define BOS combination strategies
    bos_strategies = {
        'BOS Trades (1M CC)': lambda d: d[(d['BOS/CH'] == 'BOS')],
        'BOS + Buy (1M CC)': lambda d: d[(d['BOS/CH'] == 'BOS') & (d['Direction'] == 'Buy')],
        'BOS + Sell (1M CC)': lambda d: d[(d['BOS/CH'] == 'BOS') & (d['Direction'] == 'Sell')],
        'BOS + EMA (1M CC)': lambda d: d[(d['BOS/CH'] == 'BOS') & (d['EMA'] == d['Direction'])],
        'BOS + Against EMA (1M CC)': lambda d: d[(d['BOS/CH'] == 'BOS') & (d['EMA'] != d['Direction'])],
        'BOS + HTF Buy (1M CC)': lambda d: d[(d['BOS/CH'] == 'BOS') & 
                                            (d['30M Leg'].isin(['Above H', 'Above L']))],
        'BOS + HTF Sell (1M CC)': lambda d: d[(d['BOS/CH'] == 'BOS') & 
                                             (d['30M Leg'].isin(['Below H', 'Below L']))],
        'BOS + News (1M CC)': lambda d: d[(d['BOS/CH'] == 'BOS') & (~d['News Event'].isna())],
        'BOS + Without News (1M CC)': lambda d: d[(d['BOS/CH'] == 'BOS') & (d['News Event'].isna())]
    }
    
    results = {'Idea': list(bos_strategies.keys())}
    
    # Calculate performance for each RRR level
    for rrr in [1, 2, 3]:
        rrr_results = []
        for strategy_name, filter_func in bos_strategies.items():
            # Apply base filter (valid trades) and strategy filter
            valid_trades = df[df['SL'] != df['Pullback']]
            strategy_trades = filter_func(valid_trades)
            profitable = strategy_trades[strategy_trades['TP'] >= strategy_trades['SL'] * rrr]
            
            rrr_results.append(percentage(len(profitable), len(df)))
        
        results[f'1:{rrr} RRR'] = rrr_results
    
    # Create DataFrame and sort by best 1:1 RRR performance
    df_results = pd.DataFrame(results)
    df_results['Value_num'] = df_results['1:1 RRR'].str.rstrip('%').astype(float)
    return df_results.sort_values('Value_num', ascending=False).drop(columns='Value_num').reset_index(drop=True)

# Display BOS with 1M Confirmation Candle analysis
display(HTML("<h2>BOS Breakdown with 1M Confirmation Candle Entry</h2><p>Analysis of BOS trades using 1-minute confirmation candle entry method (standard SL column)</p>"))
df_bos_1m_cc = analyze_bos_1m_cc(df)
display(df_bos_1m_cc)

# ================== BOS WITH 5M STOP ANALYSIS ==================

def analyze_bos_5m_stop(df):
    """
    Analyze Break of Structure (BOS) trades using 5M Stop entry method.
    
    This analysis evaluates BOS trades specifically when using the 5-minute stop
    entry timing, filtering out trades where SL 5M Stop is empty (0).
    
    Args:
        df (pd.DataFrame): Trading data with BOS/CH and SL 5M Stop indicators
    
    Returns:
        pd.DataFrame: BOS 5M Stop combination statistics sorted by performance
    """
    # Filter for trades with valid 5M Stop values (non-zero)
    df_with_5m_stop = df[df['SL 5M Stop'] != 0]
    
    # Define BOS combination strategies using 5M Stop
    bos_5m_strategies = {
        'BOS Trades (5M Stop)': lambda d: d[(d['BOS/CH'] == 'BOS')],
        'BOS + Buy (5M Stop)': lambda d: d[(d['BOS/CH'] == 'BOS') & (d['Direction'] == 'Buy')],
        'BOS + Sell (5M Stop)': lambda d: d[(d['BOS/CH'] == 'BOS') & (d['Direction'] == 'Sell')],
        'BOS + EMA (5M Stop)': lambda d: d[(d['BOS/CH'] == 'BOS') & (d['EMA'] == d['Direction'])],
        'BOS + Against EMA (5M Stop)': lambda d: d[(d['BOS/CH'] == 'BOS') & (d['EMA'] != d['Direction'])],
        'BOS + HTF Buy (5M Stop)': lambda d: d[(d['BOS/CH'] == 'BOS') & 
                                               (d['30M Leg'].isin(['Above H', 'Above L']))],
        'BOS + HTF Sell (5M Stop)': lambda d: d[(d['BOS/CH'] == 'BOS') & 
                                                (d['30M Leg'].isin(['Below H', 'Below L']))],
        'BOS + News (5M Stop)': lambda d: d[(d['BOS/CH'] == 'BOS') & (~d['News Event'].isna())],
        'BOS + Without News (5M Stop)': lambda d: d[(d['BOS/CH'] == 'BOS') & (d['News Event'].isna())]
    }
    
    results = {'Idea': list(bos_5m_strategies.keys())}
    
    # Calculate performance for each RRR level using SL 5M Stop
    for rrr in [1, 2, 3]:
        rrr_results = []
        for strategy_name, filter_func in bos_5m_strategies.items():
            # Apply strategy filter on trades with valid 5M Stop
            strategy_trades = filter_func(df_with_5m_stop)
            # Check profitability against SL 5M Stop instead of SL
            profitable = strategy_trades[strategy_trades['TP'] >= strategy_trades['SL 5M Stop'] * rrr]
            
            # Calculate percentage relative to total dataset
            rrr_results.append(percentage(len(profitable), len(df)))
        
        results[f'1:{rrr} RRR'] = rrr_results
    
    # Create DataFrame and sort by best 1:1 RRR performance
    df_results = pd.DataFrame(results)
    df_results['Value_num'] = df_results['1:1 RRR'].str.rstrip('%').astype(float)
    return df_results.sort_values('Value_num', ascending=False).drop(columns='Value_num').reset_index(drop=True)

# Display BOS with 5M Stop analysis
display(HTML("<h2>BOS Breakdown with 5M Stop Entry</h2><p>Analysis of BOS trades using 5-minute stop entry method (trades with SL 5M Stop = 0 are skipped)</p>"))
df_bos_5m = analyze_bos_5m_stop(df)
display(df_bos_5m)

# ================== BOS WITH 5M CONFIRMATION CANDLE ANALYSIS ==================

def analyze_bos_5m_cc(df):
    """
    Analyze Break of Structure (BOS) trades using 5M Confirmation Candle entry method.
    
    This analysis evaluates BOS trades specifically when using the 5-minute confirmation
    candle entry timing, filtering out trades where SL 5M CC is empty (0).
    
    Args:
        df (pd.DataFrame): Trading data with BOS/CH and SL 5M CC indicators
    
    Returns:
        pd.DataFrame: BOS 5M CC combination statistics sorted by performance
    """
    # Filter for trades with valid 5M CC values (non-zero)
    df_with_5m_cc = df[df['SL 5M CC'] != 0]
    
    # Define BOS combination strategies using 5M CC
    bos_5m_cc_strategies = {
        'BOS Trades (5M CC)': lambda d: d[(d['BOS/CH'] == 'BOS')],
        'BOS + Buy (5M CC)': lambda d: d[(d['BOS/CH'] == 'BOS') & (d['Direction'] == 'Buy')],
        'BOS + Sell (5M CC)': lambda d: d[(d['BOS/CH'] == 'BOS') & (d['Direction'] == 'Sell')],
        'BOS + EMA (5M CC)': lambda d: d[(d['BOS/CH'] == 'BOS') & (d['EMA'] == d['Direction'])],
        'BOS + Against EMA (5M CC)': lambda d: d[(d['BOS/CH'] == 'BOS') & (d['EMA'] != d['Direction'])],
        'BOS + HTF Buy (5M CC)': lambda d: d[(d['BOS/CH'] == 'BOS') & 
                                            (d['30M Leg'].isin(['Above H', 'Above L']))],
        'BOS + HTF Sell (5M CC)': lambda d: d[(d['BOS/CH'] == 'BOS') & 
                                             (d['30M Leg'].isin(['Below H', 'Below L']))],
        'BOS + News (5M CC)': lambda d: d[(d['BOS/CH'] == 'BOS') & (~d['News Event'].isna())],
        'BOS + Without News (5M CC)': lambda d: d[(d['BOS/CH'] == 'BOS') & (d['News Event'].isna())]
    }
    
    results = {'Idea': list(bos_5m_cc_strategies.keys())}
    
    # Calculate performance for each RRR level using SL 5M CC
    for rrr in [1, 2, 3]:
        rrr_results = []
        for strategy_name, filter_func in bos_5m_cc_strategies.items():
            # Apply strategy filter on trades with valid 5M CC
            strategy_trades = filter_func(df_with_5m_cc)
            # Check profitability against SL 5M CC instead of SL
            profitable = strategy_trades[strategy_trades['TP'] >= strategy_trades['SL 5M CC'] * rrr]
            
            # Calculate percentage relative to total dataset
            rrr_results.append(percentage(len(profitable), len(df)))
        
        results[f'1:{rrr} RRR'] = rrr_results
    
    # Create DataFrame and sort by best 1:1 RRR performance
    df_results = pd.DataFrame(results)
    df_results['Value_num'] = df_results['1:1 RRR'].str.rstrip('%').astype(float)
    return df_results.sort_values('Value_num', ascending=False).drop(columns='Value_num').reset_index(drop=True)

# Display BOS with 5M Confirmation Candle analysis
display(HTML("<h2>BOS Breakdown with 5M Confirmation Candle Entry</h2><p>Analysis of BOS trades using 5-minute confirmation candle entry method (trades with SL 5M CC = 0 are skipped)</p>"))
df_bos_5m_cc = analyze_bos_5m_cc(df)
display(df_bos_5m_cc)

# ================== COMBINED BOS ENTRY COMPARISON TABLE ==================

def create_combined_bos_table(df_1m_cc, df_5m_stop, df_5m_cc):
    """
    Create a combined table showing all BOS strategies across different entry methods.
    
    Args:
        df_1m_cc: BOS results with 1M Confirmation Candle
        df_5m_stop: BOS results with 5M Stop
        df_5m_cc: BOS results with 5M Confirmation Candle
    
    Returns:
        pd.DataFrame: Combined comparison table with multi-level columns
    """
    # Strategy base names (without entry method suffix)
    strategy_names = [
        'BOS Trades',
        'BOS + Buy',
        'BOS + Sell', 
        'BOS + EMA',
        'BOS + Against EMA',
        'BOS + HTF Buy',
        'BOS + HTF Sell',
        'BOS + News',
        'BOS + Without News'
    ]
    
    # Create combined data structure
    combined_data = []
    
    for base_name in strategy_names:
        row = {'Strategy': base_name}
        
        # Add 1M CC data
        for idx, idea in enumerate(df_1m_cc['Idea']):
            if base_name in idea:
                row['1M CC_Entry'] = '1M CC'
                row['1M CC_1:1 RRR'] = df_1m_cc.iloc[idx]['1:1 RRR']
                row['1M CC_1:2 RRR'] = df_1m_cc.iloc[idx]['1:2 RRR']
                row['1M CC_1:3 RRR'] = df_1m_cc.iloc[idx]['1:3 RRR']
                break
        
        # Add 5M Stop data
        for idx, idea in enumerate(df_5m_stop['Idea']):
            if base_name in idea:
                row['5M Stop_Entry'] = '5M Stop'
                row['5M Stop_1:1 RRR'] = df_5m_stop.iloc[idx]['1:1 RRR']
                row['5M Stop_1:2 RRR'] = df_5m_stop.iloc[idx]['1:2 RRR']
                row['5M Stop_1:3 RRR'] = df_5m_stop.iloc[idx]['1:3 RRR']
                break
        
        # Add 5M CC data
        for idx, idea in enumerate(df_5m_cc['Idea']):
            if base_name in idea:
                row['5M CC_Entry'] = '5M CC'
                row['5M CC_1:1 RRR'] = df_5m_cc.iloc[idx]['1:1 RRR']
                row['5M CC_1:2 RRR'] = df_5m_cc.iloc[idx]['1:2 RRR']
                row['5M CC_1:3 RRR'] = df_5m_cc.iloc[idx]['1:3 RRR']
                break
        
        combined_data.append(row)
    
    # Create DataFrame with specific column order
    columns = [
        'Strategy',
        '1M CC_Entry', '1M CC_1:1 RRR', '1M CC_1:2 RRR', '1M CC_1:3 RRR',
        '5M Stop_Entry', '5M Stop_1:1 RRR', '5M Stop_1:2 RRR', '5M Stop_1:3 RRR',
        '5M CC_Entry', '5M CC_1:1 RRR', '5M CC_1:2 RRR', '5M CC_1:3 RRR'
    ]
    
    combined_df = pd.DataFrame(combined_data)[columns]
    
    # Rename columns to match requested format
    combined_df.columns = [
        'Strategy',
        'Entry', '1:1 RRR', '1:2 RRR', '1:3 RRR',
        'Entry.1', '1:1 RRR.1', '1:2 RRR.1', '1:3 RRR.1',
        'Entry.2', '1:1 RRR.2', '1:2 RRR.2', '1:3 RRR.2'
    ]
    
    # Sort by best overall 1:1 RRR performance (average across all entry methods)
    def get_average_1to1(row):
        values = []
        for col in ['1:1 RRR', '1:1 RRR.1', '1:1 RRR.2']:
            if pd.notna(row[col]):
                values.append(float(row[col].rstrip('%')))
        return sum(values) / len(values) if values else 0
    
    combined_df['avg_1to1'] = combined_df.apply(get_average_1to1, axis=1)
    combined_df = combined_df.sort_values('avg_1to1', ascending=False).drop(columns='avg_1to1')
    
    return combined_df

# Create and display the combined table
display(HTML("<h2>📊 BOS Strategy Comparison Across Entry Methods</h2><p>Comprehensive comparison of all BOS strategies using different entry timing methods</p>"))
df_combined_bos = create_combined_bos_table(df_bos_1m_cc, df_bos_5m, df_bos_5m_cc)

# Style the table for better readability
styled_combined = df_combined_bos.style.set_properties(
    subset=['Strategy'],
    **{'font-weight': 'bold', 'width': '200px'}
).set_table_styles([
    {'selector': 'th', 'props': [('font-size', '11px'), ('text-align', 'center')]},
    {'selector': 'td', 'props': [('text-align', 'center')]}
])

# Apply color formatting to RRR columns
def color_value(val):
    """Color code percentage values based on performance."""
    try:
        numeric_val = float(val.rstrip('%'))
        if numeric_val >= 20:
            return 'color: green; font-weight: bold'
        elif numeric_val >= 15:
            return 'color: darkgreen'
        elif numeric_val >= 10:
            return 'color: black'
        elif numeric_val >= 5:
            return 'color: darkred'
        else:
            return 'color: red'
    except:
        return ''

# Apply color formatting to all RRR columns
# rrr_columns = [col for col in df_combined_bos.columns if 'RRR' in col]
# for col in rrr_columns:
#     styled_combined = styled_combined.map(color_value, subset=[col])

display(styled_combined)

,Idea,1:1 RRR,1:2 RRR,1:3 RRR
0,BOS Trades (1M CC),25.5%,17.3%,15.3%
1,BOS + EMA (1M CC),18.4%,12.2%,10.2%
2,BOS + Buy (1M CC),16.3%,11.2%,10.2%
3,BOS + Without News (1M CC),16.3%,11.2%,10.2%
4,BOS + HTF Buy (1M CC),14.3%,9.2%,8.2%
5,BOS + HTF Sell (1M CC),11.2%,8.2%,7.1%
6,BOS + Sell (1M CC),9.2%,6.1%,5.1%
7,BOS + News (1M CC),9.2%,6.1%,5.1%
8,BOS + Against EMA (1M CC),7.1%,5.1%,5.1%


,Idea,1:1 RRR,1:2 RRR,1:3 RRR
0,BOS Trades (5M Stop),20.4%,17.3%,10.2%
1,BOS + EMA (5M Stop),14.3%,11.2%,6.1%
2,BOS + Buy (5M Stop),13.3%,13.3%,8.2%
3,BOS + Without News (5M Stop),13.3%,11.2%,7.1%
4,BOS + HTF Sell (5M Stop),11.2%,9.2%,6.1%
5,BOS + HTF Buy (5M Stop),9.2%,8.2%,4.1%
6,BOS + Sell (5M Stop),7.1%,4.1%,2.0%
7,BOS + News (5M Stop),7.1%,6.1%,3.1%
8,BOS + Against EMA (5M Stop),6.1%,6.1%,4.1%


,Idea,1:1 RRR,1:2 RRR,1:3 RRR
0,BOS Trades (5M CC),21.4%,17.3%,12.2%
1,BOS + Buy (5M CC),14.3%,12.2%,9.2%
2,BOS + EMA (5M CC),14.3%,11.2%,8.2%
3,BOS + Without News (5M CC),14.3%,11.2%,8.2%
4,BOS + HTF Buy (5M CC),11.2%,9.2%,7.1%
5,BOS + HTF Sell (5M CC),10.2%,8.2%,5.1%
6,BOS + Sell (5M CC),7.1%,5.1%,3.1%
7,BOS + Against EMA (5M CC),7.1%,6.1%,4.1%
8,BOS + News (5M CC),7.1%,6.1%,4.1%


,Strategy,Entry,1:1 RRR,1:2 RRR,1:3 RRR,Entry.1,1:1 RRR.1,1:2 RRR.1,1:3 RRR.1,Entry.2,1:1 RRR.2,1:2 RRR.2,1:3 RRR.2
0,BOS Trades,1M CC,25.5%,17.3%,15.3%,5M Stop,20.4%,17.3%,10.2%,5M CC,21.4%,17.3%,12.2%
3,BOS + EMA,1M CC,18.4%,12.2%,10.2%,5M Stop,14.3%,11.2%,6.1%,5M CC,14.3%,11.2%,8.2%
1,BOS + Buy,1M CC,16.3%,11.2%,10.2%,5M Stop,13.3%,13.3%,8.2%,5M CC,14.3%,12.2%,9.2%
8,BOS + Without News,1M CC,16.3%,11.2%,10.2%,5M Stop,13.3%,11.2%,7.1%,5M CC,14.3%,11.2%,8.2%
5,BOS + HTF Buy,1M CC,14.3%,9.2%,8.2%,5M Stop,9.2%,8.2%,4.1%,5M CC,11.2%,9.2%,7.1%
6,BOS + HTF Sell,1M CC,11.2%,8.2%,7.1%,5M Stop,11.2%,9.2%,6.1%,5M CC,10.2%,8.2%,5.1%
2,BOS + Sell,1M CC,9.2%,6.1%,5.1%,5M Stop,7.1%,4.1%,2.0%,5M CC,7.1%,5.1%,3.1%
7,BOS + News,1M CC,9.2%,6.1%,5.1%,5M Stop,7.1%,6.1%,3.1%,5M CC,7.1%,6.1%,4.1%
4,BOS + Against EMA,1M CC,7.1%,5.1%,5.1%,5M Stop,6.1%,6.1%,4.1%,5M CC,7.1%,6.1%,4.1%


In [120]:
# ================== STRATEGY FRAMEWORK ==================

def calculate_rrr_stats(data_df, strategy_name):
    """
    Calculate comprehensive Risk-Reward Ratio statistics for a trading strategy.
    
    This function evaluates a strategy's performance across different RRR targets
    and calculates key metrics including win rate, edge over breakeven, and
    expected outcome in R-multiples.
    
    Args:
        data_df (pd.DataFrame): Filtered DataFrame containing trades for this strategy
        strategy_name (str): Name of the strategy for labeling
    
    Returns:
        pd.DataFrame: Statistics table with the following metrics:
            - Total trades: Number of trades in the strategy
            - Wins/Losses: Count of profitable vs unprofitable trades
            - Win Rate: Percentage of winning trades
            - Edge: Win rate minus breakeven rate for the RRR
            - Outcome: Net result in R-multiples (profit factor)
    """
    total_trades = len(data_df)
    
    # RRR configurations with their breakeven win rates
    # For 1:1 RRR, you need 50% wins to break even
    # For 1:2 RRR, you need 33.3% wins to break even
    # For 1:3 RRR, you need 25% wins to break even
    rrr_configs = [
        (1, 50.0),   # 1:1 RRR
        (2, 33.3),   # 1:2 RRR
        (3, 25.0),   # 1:3 RRR
    ]
    
    # Handle empty strategy results
    if total_trades == 0:
        summary_data = {
            strategy_name: ['Total trades', 'Wins', 'Losses', 'Win Rate', 'Edge', 'Outcome']
        }
        for ratio, _ in rrr_configs:
            summary_data[f'1:{ratio} RRR'] = [0, 0, 0, '0.0%', '0.0%', '0R']
        return pd.DataFrame(summary_data)
    
    # Calculate statistics for each RRR level
    summary_data = {
        strategy_name: ['Total trades', 'Wins', 'Losses', 'Win Rate', 'Edge', 'Outcome']
    }
    
    for ratio, breakeven_rate in rrr_configs:
        # Find profitable trades for this RRR
        profitable = data_df[data_df['TP'] > ratio * data_df['SL']]
        wins = len(profitable)
        losses = total_trades - wins
        win_rate = (wins / total_trades) * 100
        
        # Edge is how much better we perform than breakeven
        edge = win_rate - breakeven_rate
        
        # Calculate expected outcome in R-multiples
        # Winners make 'ratio' R, losers lose 1R
        outcome = (wins * ratio) - losses
        
        summary_data[f'1:{ratio} RRR'] = [
            total_trades,
            wins,
            losses,
            f"{win_rate:.1f}%",
            f"{edge:.1f}%",
            f"{outcome}R"
        ]
    
    return pd.DataFrame(summary_data)


class Strategy:
    """
    Encapsulates a trading strategy with its filter logic and metadata.
    
    Attributes:
        name (str): Strategy identifier
        filter_func (callable): Function that filters trades based on strategy rules
        description (str): Human-readable description of the strategy
    """
    
    def __init__(self, name, filter_func, description=""):
        """
        Initialize a trading strategy.
        
        Args:
            name (str): Strategy name
            filter_func (callable): Lambda or function that takes df and returns filtered df
            description (str): Optional description of the strategy
        """
        self.name = name
        self.filter_func = filter_func
        self.description = description
    
    def apply(self, df):
        """
        Apply the strategy filter to a dataframe of trades.
        
        Args:
            df (pd.DataFrame): Trading data
            
        Returns:
            pd.DataFrame: Filtered trades matching strategy criteria
        """
        return self.filter_func(df)

In [121]:
# ================== STRATEGY DEFINITIONS ==================

# Initialize base strategy list
strategies = [
    Strategy(
        "Plain Strategy",
        lambda df: df,
        "Baseline: All trades without any filtering"
    )
]

In [122]:
# ================== STRATEGY FACTORY ==================

def create_strategy_library():
    """
    Create a comprehensive library of trading strategies for backtesting.
    
    This function generates strategies across multiple categories:
    1. Technical Indicators (EMA, BOS/CH)
    2. Risk Management (Stop Loss levels)
    3. Market Structure (Trend alignment)
    4. Time-based (Weekdays, News events)
    5. Combined filters (Multi-factor strategies)
    
    Returns:
        list: List of Strategy objects ready for backtesting
    """
    strategy_configs = []
    
    # ========== TECHNICAL INDICATOR STRATEGIES ==========
    # EMA (Exponential Moving Average) based strategies
    strategy_configs.extend([
        ("EMA + Trade Direction", 
         lambda df: df[df['EMA'] == df['Direction']], 
         "Trade with EMA trend"),
        ("EMA + Opposite Trade Direction", 
         lambda df: df[df['EMA'] != df['Direction']], 
         "Counter-trend trades"),
        ("EMA + BOS", 
         lambda df: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS')], 
         "Trend + Break of Structure"),
        ("EMA + CH", 
         lambda df: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'CH')], 
         "Trend + Change of Character"),
    ])
    
    # Market Structure strategies
    strategy_configs.extend([
        ("BOS", 
         lambda df: df[df['BOS/CH'] == 'BOS'], 
         "Break of Structure trades only"),
        ("CH", 
         lambda df: df[df['BOS/CH'] == 'CH'], 
         "Change of Character trades only"),
    ])
    
    # ========== RISK MANAGEMENT STRATEGIES ==========
    # Stop Loss size filtering
    strategy_configs.extend([
        ("Conservative: SL <= 2 pips", 
         lambda df: df[df['SL'] <= 2], 
         "Very tight stop losses"),
        ("Moderate Risk: SL 3-6 pips", 
         lambda df: df[(df['SL'] >= 3) & (df['SL'] <= 6)], 
         "Medium stop losses"),
        ("Aggressive: SL >= 7 pips", 
         lambda df: df[df['SL'] >= 7], 
         "Wide stop losses"),
    ])
    
    # Risk-adjusted market structure combinations
    strategy_configs.extend([
        ("BOS + Conservative SL", 
         lambda df: df[(df['BOS/CH'] == 'BOS') & (df['SL'] <= 2)], 
         "BOS with tight stops"),
        ("BOS + Moderate SL", 
         lambda df: df[(df['BOS/CH'] == 'BOS') & (df['SL'] >= 3) & (df['SL'] <= 6)], 
         "BOS with medium stops"),
    ])
    
    # ========== TIME-BASED STRATEGIES ==========
    # Day of week analysis
    # weekdays = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
    # for day in weekdays:
    #     strategy_configs.append(
    #         (f"{day} Only", 
    #          lambda df, d=day: df[df['Weekday'] == d], 
    #          f"Trades on {day}")
    #     )
    
    # ========== HIGHER TIMEFRAME TREND ==========
    # 30-minute timeframe trend alignment
    strategy_configs.extend([
        ("With 30M Trend", 
         lambda df: df[(df['30M Leg'].isin(['Above H', 'Above L']) & (df['Direction'] == 'Buy')) | (df['30M Leg'].isin(['Below H', 'Below L']) & (df['Direction'] == 'Sell'))], 
         "Higher timeframe uptrend"),
    ])
    
    # ========== NEWS EVENT STRATEGIES ==========
    strategy_configs.extend([
        ("No News Events", 
         lambda df: df[df['News Event'].isna()], 
         "Avoid news volatility"),
        ("News Event Present", 
         lambda df: df[~df['News Event'].isna()], 
         "Trade during news"),
        ("News in 2+ Hours", 
         lambda df: df[(~df['News Event'].isna()) & (df['Hours Until News'] >= 2)], 
         "Trade before news impact"),
    ])
    
    # ========== COMBINED MULTI-FACTOR STRATEGIES ==========
    # These combine multiple indicators for potentially higher probability setups
    strategy_configs.extend([
        ("EMA + BOS + Low Risk", 
         lambda df: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS') & (df['SL'] <= 3)], 
         "Triple confirmation setup"),
        ("BOS + Bullish 30M + No News", 
         lambda df: df[(df['BOS/CH'] == 'BOS') & 
                      (df['30M Leg'].isin(['Above H', 'Above L'])) & 
                      (df['News Event'].isna())], 
         "Clean trend continuation"),
        ("Strong Alignment + EMA", 
         lambda df: df[((df['30M Leg'] == 'Above H') & (df['Direction'] == 'Buy') & 
                       (df['EMA'] == 'Buy')) | 
                      ((df['30M Leg'] == 'Below L') & (df['Direction'] == 'Sell') & 
                       (df['EMA'] == 'Sell'))], 
         "Full trend confluence"),
    ])
    
    # Convert configurations to Strategy objects
    return [Strategy(name, func, desc) for name, func, desc in strategy_configs]

# Add all strategies to the main list
strategies.extend(create_strategy_library())

# ================== STRATEGY EVALUATION ==================

def evaluate_all_strategies(df, strategies):
    """
    Run backtesting on all strategies and compile results.
    
    Args:
        df (pd.DataFrame): Trading data
        strategies (list): List of Strategy objects
        
    Returns:
        dict: Dictionary mapping strategy names to their performance DataFrames
    """
    strategy_results = {}
    
    for strategy in strategies:
        # Apply strategy filter
        filtered_df = strategy.apply(df)
        
        # Calculate RRR statistics
        summary_df = calculate_rrr_stats(filtered_df, strategy.name)
        
        # Store results
        strategy_results[strategy.name] = summary_df
    
    return strategy_results

def get_top_strategies(strategy_results, rrr_column, top_n=5):
    """
    Extract top performing strategies for a specific RRR.
    
    Args:
        strategy_results (dict): Dictionary of strategy results
        rrr_column (str): Column name for RRR (e.g., '1:2 RRR')
        top_n (int): Number of top strategies to return
        
    Returns:
        pd.DataFrame: Top strategies ranked by outcome
    """
    strategy_performance = []
    
    for strategy_name, df in strategy_results.items():
        # Extract performance metrics
        total_trades = df[rrr_column].iloc[0]
        wins = df[rrr_column].iloc[1]
        losses = df[rrr_column].iloc[2]
        win_rate = df[rrr_column].iloc[3]
        edge = df[rrr_column].iloc[4]
        outcome_str = df[rrr_column].iloc[5]
        
        # Parse outcome value for sorting
        outcome = int(outcome_str.replace('R', ''))
        
        strategy_performance.append({
            'Strategy': strategy_name,
            'Trades': total_trades,
            'Wins': wins,
            'Losses': losses,
            'Win Rate': win_rate,
            'Edge': edge,
            'Outcome': outcome_str,
            'outcome_value': outcome
        })
    
    # Sort by outcome and get top N
    top_strategies = sorted(
        strategy_performance, 
        key=lambda x: x['outcome_value'], 
        reverse=True
    )[:top_n]
    
    # Remove sorting key from display
    for strat in top_strategies:
        del strat['outcome_value']
    
    return pd.DataFrame(top_strategies)

# ================== RUN BACKTESTING ==================

# Evaluate all strategies
print(f"Evaluating {len(strategies)} strategies...")
strategy_results = evaluate_all_strategies(df, strategies)

# Display top performing strategies for each RRR
display(HTML("<h2>🏆 TOP Performing Strategies</h2>"))

rrr_configs = [
    ('1:1 RRR', '1:1'),
    ('1:2 RRR', '1:2'),
    ('1:3 RRR', '1:3'),
]

for rrr_column, rrr_label in rrr_configs:
    display(HTML(f"<h3>Best {rrr_label} Strategies</h3>"))
    
    # Get and display top 5 strategies
    top_5_df = get_top_strategies(strategy_results, rrr_column, top_n=5)
    top_5_df = top_5_df.rename(columns={'Strategy': f'Top {rrr_label} Strategies'})
    
    # Style the table for better readability
    styled_df = top_5_df.style.set_properties(
        subset=[f'Top {rrr_label} Strategies'], 
        **{'width': '300px', 'font-weight': 'bold'}
    ).set_properties(
        subset=['Outcome'],
        **{'color': 'green', 'font-weight': 'bold'}
    )
    
    display(styled_df)
    print()  # Add spacing

Evaluating 19 strategies...


,Top 1:1 Strategies,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,BOS,61,33,28,54.1%,4.1%,5R
1,EMA + Opposite Trade Direction,38,21,17,55.3%,5.3%,4R
2,No News Events,54,29,25,53.7%,3.7%,4R
3,BOS + Bullish 30M + No News,20,12,8,60.0%,10.0%,4R
4,Aggressive: SL >= 7 pips,17,10,7,58.8%,8.8%,3R


,Top 1:2 Strategies,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,BOS,61,24,37,39.3%,6.0%,11R
1,News in 2+ Hours,26,12,14,46.2%,12.9%,10R
2,With 30M Trend,60,22,38,36.7%,3.4%,6R
3,Strong Alignment + EMA,21,9,12,42.9%,9.6%,6R
4,EMA + BOS + Low Risk,16,7,9,43.8%,10.5%,5R


,Top 1:3 Strategies,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,BOS,61,22,39,36.1%,11.1%,27R
1,With 30M Trend,60,19,41,31.7%,6.7%,16R
2,Plain Strategy,98,28,70,28.6%,3.6%,14R
3,EMA + Opposite Trade Direction,38,13,25,34.2%,9.2%,14R
4,EMA + BOS + Low Risk,16,7,9,43.8%,18.8%,12R


In [123]:
# ================== DETAILED STRATEGY RESULTS ==================

def display_detailed_results(strategy_results, display_all=False):
    """
    Display detailed results for all or selected strategies.
    
    Args:
        strategy_results (dict): Dictionary of strategy performance DataFrames
        display_all (bool): If True, show all strategies; if False, show only profitable ones
    """
    if display_all:
        display(HTML(f"<h2>📊 All Strategy Results ({len(strategy_results)} strategies)</h2>"))
        strategies_to_display = strategy_results.items()
    else:
        # Filter for profitable strategies (positive outcome at 1:1 RRR)
        profitable_strategies = []
        for name, df in strategy_results.items():
            outcome_str = df['1:1 RRR'].iloc[5]
            outcome = int(outcome_str.replace('R', ''))
            if outcome > 0:
                profitable_strategies.append((name, df))
        
        display(HTML(f"<h2>💰 Profitable Strategies ({len(profitable_strategies)} of {len(strategy_results)})</h2>"))
        strategies_to_display = profitable_strategies
    
    # Display each strategy's results
    for strategy_name, summary_df in strategies_to_display:
        # Style the DataFrame for better readability
        styled_df = summary_df.style.set_properties(
            subset=[strategy_name], 
            **{'width': '300px', 'font-weight': 'bold'}
        ).set_properties(
            **{'text-align': 'center'}
        )
        
        # Highlight positive outcomes in green, negative in red
        for col in ['1:1 RRR', '1:2 RRR', '1:3 RRR']:
            outcome_str = summary_df[col].iloc[5]
            outcome = int(outcome_str.replace('R', ''))
            color = 'green' if outcome > 0 else 'red' if outcome < 0 else 'gray'
            styled_df = styled_df.set_properties(
                subset=[col],
                **{'color': color if summary_df.index[5] == 'Outcome' else 'white'}
            )
        
        display(styled_df)
        print()  # Add spacing

# Display only profitable strategies by default
display_detailed_results(strategy_results, display_all=False)

# Option to see all strategies (uncomment to view)
# display_detailed_results(strategy_results, display_all=True)

,Plain Strategy,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,98,98,98
1,Wins,50,33,28
2,Losses,48,65,70
3,Win Rate,51.0%,33.7%,28.6%
4,Edge,1.0%,0.4%,3.6%
5,Outcome,2R,1R,14R


,EMA + Opposite Trade Direction,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,38,38,38
1,Wins,21,14,13
2,Losses,17,24,25
3,Win Rate,55.3%,36.8%,34.2%
4,Edge,5.3%,3.5%,9.2%
5,Outcome,4R,4R,14R


,BOS,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,61,61,61
1,Wins,33,24,22
2,Losses,28,37,39
3,Win Rate,54.1%,39.3%,36.1%
4,Edge,4.1%,6.0%,11.1%
5,Outcome,5R,11R,27R


,Moderate Risk: SL 3-6 pips,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,53,53,53
1,Wins,27,16,13
2,Losses,26,37,40
3,Win Rate,50.9%,30.2%,24.5%
4,Edge,0.9%,-3.1%,-0.5%
5,Outcome,1R,-5R,-1R


,Aggressive: SL >= 7 pips,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,17,17,17
1,Wins,10,5,4
2,Losses,7,12,13
3,Win Rate,58.8%,29.4%,23.5%
4,Edge,8.8%,-3.9%,-1.5%
5,Outcome,3R,-2R,-1R


...

,With 30M Trend,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,60,60,60
1,Wins,31,22,19
2,Losses,29,38,41
3,Win Rate,51.7%,36.7%,31.7%
4,Edge,1.7%,3.4%,6.7%
5,Outcome,2R,6R,16R


,No News Events,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,54,54,54
1,Wins,29,18,16
2,Losses,25,36,38
3,Win Rate,53.7%,33.3%,29.6%
4,Edge,3.7%,0.0%,4.6%
5,Outcome,4R,0R,10R


,News in 2+ Hours,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,26,26,26
1,Wins,14,12,9
2,Losses,12,14,17
3,Win Rate,53.8%,46.2%,34.6%
4,Edge,3.8%,12.9%,9.6%
5,Outcome,2R,10R,10R


,BOS + Bullish 30M + No News,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,20,20,20
1,Wins,12,8,8
2,Losses,8,12,12
3,Win Rate,60.0%,40.0%,40.0%
4,Edge,10.0%,6.7%,15.0%
5,Outcome,4R,4R,12R


,Strong Alignment + EMA,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,21,21,21
1,Wins,11,9,6
2,Losses,10,12,15
3,Win Rate,52.4%,42.9%,28.6%
4,Edge,2.4%,9.6%,3.6%
5,Outcome,1R,6R,3R
